# Taller: robot omnidireccional en ROS 2

**Robot:** Clearpath Dingo-O (4 ruedas mecanum) · **Simulador:** Gazebo Harmonic · **ROS 2:** Jazzy · **Mundo:** laboratorio CoRAL

En este taller vas a hacer que el robot recorra una trayectoria escribiendo **tú** el modelo matemático. El simulador y los nodos de ROS 2 ya están listos: tu código va en un solo archivo.

| Nivel | Qué recibe el robot | Qué programas tú | Qué aprendes |
|---|---|---|---|
| **1** | velocidades del cuerpo `(vx, vy, ω)` | `mundo_a_robot()` | modelo cinemático: marco del mundo vs marco del robot |
| **2** | velocidad de **cada rueda** | `cinematica_inversa()` | cinemática inversa de ruedas mecanum |
| **3** | velocidad de cada rueda | `controlador()` | control en lazo cerrado |

**Cómo usar este notebook:** cada nivel tiene teoría, una celda donde escribes la función, una celda que la **prueba** sin ROS, y las instrucciones para correrla en el simulador.

## 0. Dónde está cada cosa

```
~/practica_03/src/taller_omni/
├── taller_omni/
│   ├── modelo.py        ← EL ÚNICO ARCHIVO QUE EDITAS
│   ├── nivel1.py        nodos de ROS 2 ya escritos:
│   ├── nivel2.py        llaman a las funciones de modelo.py
│   ├── nivel3.py
│   ├── graficar.py      graba y dibuja deseada vs real
│   ├── reiniciar.py     devuelve el robot al inicio
│   └── pose_real.py     publica la pose real del robot (desde Gazebo)
├── notebook/Taller_Robot_Omnidireccional.ipynb   ← este notebook
├── launch/taller.launch.py
├── README.md            instrucciones del taller
└── setup.py             los nodos ya están registrados aquí (entry_points)
```

Todos trabajan con **el mismo robot** (mismas medidas); lo que cambia es la **trayectoria**, que depende de tu número de lista.

En `setup.py` ya está registrado cada nodo, por eso puedes ejecutarlos con `ros2 run taller_omni <nombre>`:

```python
entry_points={
    'console_scripts': [
        'nivel1 = taller_omni.nivel1:main',
        'nivel2 = taller_omni.nivel2:main',
        ...
```

El workspace se compila con `--symlink-install`, así que **después de editar `modelo.py` no hace falta recompilar**: guarda y vuelve a ejecutar el nodo.

Para compilar la primera vez (solo una vez):

```bash
cd ~/practica_03
colcon build --symlink-install
source install/setup.bash
```

## 1. Levantar el simulador

Vas a usar **tres terminales**. En cada una, primero:

```bash
cd ~/practica_03 && source install/setup.bash
```

| Terminal | Comando | Para qué |
|---|---|---|
| 1 | `ros2 launch taller_omni taller.launch.py nivel:=1` | simulador (déjalo abierto) |
| 2 | `ros2 run taller_omni graficar` | graba y guarda el gráfico al terminar |
| 3 | `ros2 run taller_omni nivel1` | ejecuta **tu** trayectoria |

El robot aparece dentro de la **jaula de drones** (el área libre más grande del laboratorio). Antes de cada intento devuélvelo al inicio con:

```bash
ros2 run taller_omni reiniciar
```

Los gráficos quedan en `~/taller_resultados/`.

Mientras no escribas tu trayectoria, los nodos pueden usar la de **ejemplo**:

```bash
ros2 run taller_omni nivel1 --ros-args -p ejemplo:=true
```

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Medidas del Dingo-O (las mismas de modelo.py)
R = 0.05      # radio de rueda [m]
LX = 0.285    # centro -> eje de ruedas, adelante/atrás [m]
LY = 0.1854   # centro -> centro de rueda, lateral [m]
DURACION = 30.0

## 2. El robot

Convención de ROS: en el **marco del robot**, $x$ apunta hacia adelante, $y$ hacia la izquierda y $\omega$ es positivo girando en sentido antihorario.

| Parámetro | Símbolo | Valor |
|---|---|---|
| Radio de rueda | $r$ | 0.05 m |
| Distancia longitudinal | $l_x$ | 0.285 m |
| Distancia lateral | $l_y$ | 0.1854 m |
| Velocidad máxima lineal | | 1.3 m/s |
| Velocidad máxima de giro | | 4 rad/s |

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.add_patch(plt.Rectangle((-0.34, -0.22), 0.68, 0.44, fc='#ffcc80', ec='k'))
for nombre, sx, sy in [('fl', 1, 1), ('fr', 1, -1), ('rl', -1, 1), ('rr', -1, -1)]:
    ax.add_patch(plt.Rectangle((sx * LX - R, sy * LY - 0.025), 2 * R, 0.05, fc='0.3'))
    ax.text(sx * LX, sy * (LY + 0.08), nombre, ha='center', va='center', fontsize=12)
ax.annotate('', (0.3, 0), (0, 0), arrowprops=dict(arrowstyle='->', color='r', lw=2))
ax.annotate('', (0, 0.3), (0, 0), arrowprops=dict(arrowstyle='->', color='g', lw=2))
ax.text(0.31, -0.04, 'x (adelante)', color='r')
ax.text(0.02, 0.31, 'y (izquierda)', color='g')
ax.plot([0, LX], [-0.32, -0.32], 'k|-'); ax.text(LX / 2, -0.36, '$l_x$', ha='center')
ax.plot([-0.42, -0.42], [0, LY], 'k_-'); ax.text(-0.47, LY / 2, '$l_y$', va='center')
ax.set_xlim(-0.55, 0.6); ax.set_ylim(-0.45, 0.45); ax.set_aspect('equal'); ax.grid(alpha=0.3)
ax.set_title('Dingo-O visto desde arriba')
plt.show()

## 3. Tu trayectoria

Una trayectoria dice **dónde** debe estar el robot en cada instante y **con qué velocidad** moverse:

$$x_d(t),\; y_d(t),\; \theta_d(t) \qquad \dot x_d(t),\; \dot y_d(t),\; \dot\theta_d(t)$$

### Ejemplo resuelto: círculo girando

Ya está en `modelo.py` como `trayectoria_ejemplo`: un círculo de radio 1 m que empieza en el origen **mientras el robot da una vuelta completa sobre sí mismo**. Un robot diferencial no puede hacer esto, porque debe mirar siempre hacia donde avanza. El omnidireccional, sí.

$$x_d = \rho\sin(\Omega t),\quad y_d = \rho\,(1-\cos(\Omega t)),\quad \theta_d = \Omega t,\qquad \Omega = \frac{2\pi}{T}$$

Las velocidades son las derivadas: $\dot x_d = \rho\,\Omega\cos(\Omega t)$, $\dot y_d = \rho\,\Omega\sin(\Omega t)$, $\dot\theta_d = \Omega$.

In [ ]:
def trayectoria_ejemplo(t):
    radio = 1.0
    w = 2 * math.pi / DURACION
    x = radio * math.sin(w * t)
    y = radio * (1 - math.cos(w * t))
    theta = w * t
    dx = radio * w * math.cos(w * t)
    dy = radio * w * math.sin(w * t)
    dtheta = w
    return x, y, theta, dx, dy, dtheta


def dibujar(tray, titulo):
    ts = np.linspace(0, DURACION, 600)
    P = np.array([tray(t) for t in ts])
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([-2.0, 2.0, 2.0, -2.0, -2.0], [-0.925, -0.925, 2.875, 2.875, -0.925], color='0.6', label='jaula')
    ax.plot(P[:, 0], P[:, 1], '--', label='camino')
    for k in range(0, 600, 50):   # flechas: hacia donde MIRA el robot
        x, y, th = P[k, :3]
        ax.arrow(x, y, 0.2 * math.cos(th), 0.2 * math.sin(th), head_width=0.06, color='r')
    ax.plot(0, 0, 'ko', label='inicio')
    ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.set_title(titulo)
    plt.show()


dibujar(trayectoria_ejemplo, 'Ejemplo (flechas = orientación)')

### Tu asignación

Cada persona tiene una trayectoria distinta: una **forma** y un **modo de orientación**, según su número de lista $n$:

$$\text{forma} = ((n-1) \bmod 6) + 1, \qquad \text{modo} = \text{A, B o C según } \left\lfloor \tfrac{n-1}{6} \right\rfloor \bmod 3$$

| Forma | Descripción (empieza en el origen, sentido antihorario) |
|---|---|
| 1 | **Círculo** de radio 0.8 m con centro en (0, 0.8) |
| 2 | **Elipse** de centro (0, 0.7), semieje 1.2 m en $x$ y 0.7 m en $y$ |
| 3 | **Ocho vertical**: $x$ oscila 2 veces con amplitud 0.6 m mientras $y$ sube de 0 a 1.8 m y vuelve una vez |
| 4 | **Cuadrado** de 1.2 m: (0,0) → (1.2,0) → (1.2,1.2) → (0,1.2) → (0,0) |
| 5 | **Triángulo equilátero** de 1.4 m: (0,0) → (1.4,0) → (0.7, 1.212) → (0,0) |
| 6 | **Rombo**: (0,0) → (0.8,0.9) → (0,1.8) → (−0.8,0.9) → (0,0) |

| Modo | Orientación $\theta_d(t)$ |
|---|---|
| A | fija: $\theta_d = 0$ |
| B | una vuelta completa antihoraria: $\theta_d = 2\pi t/T$ |
| C | oscila ±45°: $\theta_d = \frac{\pi}{4}\sin(2\pi t/T)$ |

Reglas para todas: dura $T$ = `DURACION` = 30 s, empieza y termina en el origen, y en los polígonos la velocidad es **constante** (cada lado dura lo mismo).

✏️ Escribe tu número de lista y mira qué te toca:

In [ ]:
NUMERO_DE_LISTA = 1   # <- cambia esto

forma = (NUMERO_DE_LISTA - 1) % 6 + 1
modo = 'ABC'[((NUMERO_DE_LISTA - 1) // 6) % 3]
print(f'Te toca: forma {forma}, modo {modo}')

✏️ **Escribe tu trayectoria.** Devuelve `(x, y, theta, dx, dy, dtheta)`.

Pistas:
- Curvas (formas 1 a 3): escribe $x(t)$ e $y(t)$ con senos y cosenos, como en el ejemplo, y deriva a mano.
- Polígonos (formas 4 a 6): averigua en qué lado estás con `k = int(t // (T / lados))` e interpola linealmente entre dos vértices. La velocidad es constante en cada lado.

In [ ]:
def trayectoria(t):
    # >>> ESCRIBE AQUI <<<
    raise NotImplementedError

In [ ]:
# Prueba (no la edites)
def probar_trayectoria():
    try:
        trayectoria(0.0)
    except NotImplementedError:
        print('⏳ Aún no escribes trayectoria()')
        return False
    T = DURACION
    ini, fin = trayectoria(0.0), trayectoria(T - 1e-9)
    if not np.allclose(ini[:3], 0, atol=1e-6):
        print(f'❌ En t=0 debe estar en el origen con theta=0 y está en {tuple(np.round(ini[:3], 3))}')
        return False
    if not np.allclose(fin[:2], 0, atol=1e-3):
        print(f'❌ En t=T debe volver al origen y termina en {tuple(np.round(fin[:2], 3))}')
        return False
    # Las derivadas deben coincidir con la pendiente numérica (salvo en las esquinas)
    h, malos, n = 1e-5, 0, 0
    for t in np.linspace(0.01, T - 0.01, 997):
        a, b, p = trayectoria(t - h), trayectoria(t + h), trayectoria(t)
        num = [(b[i] - a[i]) / (2 * h) for i in range(3)]
        n += 1
        if not np.allclose(num, p[3:], atol=1e-3):
            malos += 1
    if malos > 0.02 * n:
        print(f'❌ dx, dy o dtheta no son la derivada de x, y, theta ({malos} de {n} puntos no coinciden)')
        return False
    P = np.array([trayectoria(t)[:2] for t in np.linspace(0, T, 600)])
    if P[:, 0].min() < -1.55 or P[:, 0].max() > 1.55 or P[:, 1].min() < -0.45 or P[:, 1].max() > 2.4:
        print('❌ La trayectoria se sale de la zona segura de la jaula')
        return False
    print('✅ Trayectoria válida. Compara el dibujo con tu asignación y cópiala a modelo.py')
    return True


if probar_trayectoria():
    dibujar(trayectoria, f'Tu trayectoria (forma {forma}, modo {modo})')

# El resto del notebook usa tu trayectoria si ya está lista; si no, la de ejemplo
try:
    trayectoria(0.0)
    tray = trayectoria
except NotImplementedError:
    tray = trayectoria_ejemplo
ts = np.linspace(0, DURACION, 600)
X = np.array([tray(t) for t in ts])

## 4. Nivel 1 — Del marco del mundo al marco del robot

La trayectoria está en el **marco del mundo**, pero el robot recibe por `cmd_vel` velocidades en **su propio marco**. Si el robot está girado un ángulo $\theta$, "ir hacia $+x$ del mundo" no es lo mismo que "ir hacia adelante".

La relación entre ambos marcos es una rotación:

$$\begin{bmatrix}\dot x\\ \dot y\end{bmatrix} = \underbrace{\begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}}_{R(\theta)} \begin{bmatrix}v_x\\ v_y\end{bmatrix}, \qquad \dot\theta = \omega$$

Tú necesitas lo contrario: dadas $(\dot x, \dot y, \dot\theta)$, obtener $(v_x, v_y, \omega)$. Como $R(\theta)$ es una rotación, su inversa es su transpuesta:

$$\begin{bmatrix}v_x\\ v_y\end{bmatrix} = R(\theta)^T \begin{bmatrix}\dot x\\ \dot y\end{bmatrix}, \qquad \omega = \dot\theta$$

✏️ **Escribe la función** (desarrolla $R(\theta)^T$ a mano: son dos líneas).

In [ ]:
def mundo_a_robot(theta, dx, dy, dtheta):
    # >>> ESCRIBE AQUI <<<
    raise NotImplementedError

In [ ]:
# Prueba (no la edites)
def probar_nivel1():
    casos = [
        # (theta, dx, dy, dtheta) -> (vx, vy, w)        explicación
        ((0.0, 1, 0, 0), (1, 0, 0), 'sin girar: el mundo y el robot coinciden'),
        ((math.pi / 2, 1, 0, 0), (0, -1, 0), 'mirando a +y, ir a +x del mundo es ir a la DERECHA del robot'),
        ((math.pi, 0, 1, 0.3), (0, -1, 0.3), 'de espaldas: +y del mundo es la derecha del robot'),
    ]
    try:
        for entrada, esperado, explicacion in casos:
            obtenido = mundo_a_robot(*entrada)
            if not np.allclose(obtenido, esperado, atol=1e-9):
                print(f'❌ {explicacion}\n   entrada {entrada}: esperado {esperado}, obtenido {tuple(np.round(obtenido, 3))}')
                return
    except NotImplementedError:
        print('⏳ Aún no escribes mundo_a_robot()')
        return
    print('✅ Nivel 1 correcto. Copia la función a modelo.py')

probar_nivel1()

In [ ]:
# ¿Qué velocidades recibe el robot durante tu trayectoria?
try:
    V = np.array([mundo_a_robot(X[k, 2], *X[k, 3:]) for k in range(len(ts))])
    plt.figure(figsize=(8, 3))
    plt.plot(ts, V[:, 0], label='$v_x$'); plt.plot(ts, V[:, 1], label='$v_y$'); plt.plot(ts, V[:, 2], label='$\\omega$')
    plt.xlabel('t [s]'); plt.legend(); plt.grid(alpha=0.3); plt.title('Comando en el marco del robot')
    plt.show()
except NotImplementedError:
    print('⏳ Primero escribe mundo_a_robot()')

### En el simulador

1. Copia `trayectoria` y `mundo_a_robot` en `modelo.py` (reemplaza cada `raise NotImplementedError`).
2. Terminal 1: `ros2 launch taller_omni taller.launch.py nivel:=1`
3. Terminal 2: `ros2 run taller_omni graficar`
4. Terminal 3: `ros2 run taller_omni nivel1` (o primero con `--ros-args -p ejemplo:=true` para probar con el círculo de ejemplo)

**Preguntas**
- En el modo B, ¿por qué $v_x$ y $v_y$ cambian todo el tiempo aunque el robot recorra la figura a velocidad constante?
- Cambia temporalmente tu función para que ignore $\theta$ (usa `theta = 0`). ¿Qué hace el robot? ¿Por qué?
- El robot no sigue el círculo perfectamente. ¿De dónde viene el error si tu fórmula es exacta? *(lo resolveremos en el nivel 3)*

## 5. Nivel 2 — Cinemática inversa de las ruedas mecanum

Hasta ahora el controlador de Clearpath convertía $(v_x, v_y, \omega)$ en la velocidad de cada rueda. **En el nivel 2 lo quitamos**: el robot recibe directamente la velocidad angular de cada rueda y esa conversión la haces tú.

Cada rueda mecanum tiene rodillos a 45°, por eso puede generar fuerza en diagonal. Combinando las cuatro se logra cualquier movimiento en el plano:

$$\begin{bmatrix}\omega_{fl}\\ \omega_{fr}\\ \omega_{rl}\\ \omega_{rr}\end{bmatrix} = \frac{1}{r}\begin{bmatrix}1 & -1 & -(l_x+l_y)\\ 1 & 1 & (l_x+l_y)\\ 1 & 1 & -(l_x+l_y)\\ 1 & -1 & (l_x+l_y)\end{bmatrix}\begin{bmatrix}v_x\\ v_y\\ \omega\end{bmatrix}$$

Interpretación de cada columna:
- **$v_x$** (avanzar): las 4 ruedas giran igual.
- **$v_y$** (lateral izquierda): las ruedas de cada diagonal giran igual y las dos diagonales en sentido contrario.
- **$\omega$** (girar): las ruedas izquierdas hacia atrás y las derechas hacia adelante. El brazo de palanca es $l_x + l_y$.

✏️ **Escribe la función.** Devuelve las cuatro velocidades en el orden `(fl, fr, rl, rr)`.

In [ ]:
def cinematica_inversa(vx, vy, w):
    # >>> ESCRIBE AQUI <<<
    raise NotImplementedError

In [ ]:
# Cinemática DIRECTA (ya resuelta): de las ruedas al movimiento del robot
def cinematica_directa(w_fl, w_fr, w_rl, w_rr):
    vx = R / 4 * (w_fl + w_fr + w_rl + w_rr)
    vy = R / 4 * (-w_fl + w_fr + w_rl - w_rr)
    w = R / (4 * (LX + LY)) * (-w_fl + w_fr - w_rl + w_rr)
    return vx, vy, w


# Prueba (no la edites): si tu inversa es correcta, directa(inversa(v)) == v
def probar_nivel2():
    try:
        assert np.allclose(cinematica_inversa(0.2, 0, 0), (4, 4, 4, 4)), 'avanzar 0.2 m/s: todas las ruedas a 4 rad/s'
        for v in [(0.2, 0, 0), (0, 0.2, 0), (0, 0, 0.5), (0.3, -0.1, 0.4), (-0.2, 0.25, -1.0)]:
            vuelta = cinematica_directa(*cinematica_inversa(*v))
            assert np.allclose(vuelta, v), f'para {v} la directa devuelve {tuple(np.round(vuelta, 3))}: revisa los signos'
    except NotImplementedError:
        print('⏳ Aún no escribes cinematica_inversa()')
        return
    except AssertionError as e:
        print('❌', e)
        return
    print('✅ Nivel 2 correcto. Copia la función a modelo.py')

probar_nivel2()

In [ ]:
# Qué hace cada rueda en los movimientos básicos
movimientos = {'adelante': (0.2, 0, 0), 'izquierda': (0, 0.2, 0), 'diagonal': (0.2, 0.2, 0),
               'girar': (0, 0, 0.5)}
try:
    fig, axs = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
    for ax, (nombre, v) in zip(axs, movimientos.items()):
        ax.bar(['fl', 'fr', 'rl', 'rr'], cinematica_inversa(*v), color=['C0', 'C1', 'C2', 'C3'])
        ax.axhline(0, color='k', lw=0.8); ax.set_title(nombre); ax.grid(alpha=0.3)
    axs[0].set_ylabel('rad/s')
    plt.show()
except NotImplementedError:
    print('⏳ Primero escribe cinematica_inversa()')

### En el simulador

1. Copia `cinematica_inversa` a `modelo.py`.
2. **Reinicia el simulador en modo rueda**: cierra la terminal 1 (Ctrl+C) y lanza `ros2 launch taller_omni taller.launch.py nivel:=2`. Espera el mensaje *"Controlador por rueda activo"*.
3. Terminal 2: `ros2 run taller_omni graficar` · Terminal 3: `ros2 run taller_omni nivel2`

> En modo rueda el panel de teleoperación de Gazebo deja de mover el robot: ahora las ruedas solo te obedecen a ti.

**Preguntas**
- Cambia el signo de una columna de la matriz. ¿Qué movimiento se rompe?
- ¿Por qué en la diagonal dos ruedas quedan en cero?

## 6. Nivel 3 — Lazo cerrado

En los niveles 1 y 2 el robot va en **lazo abierto**: manda velocidades y "confía" en que llegó. Las ruedas patinan un poco, el robot acelera con retraso... y el error se acumula.

Solución: medir dónde está realmente el robot y corregir. Se suma un término **proporcional al error** a la velocidad deseada (*feedforward + P*):

$$\dot x = \dot x_d + K_p\,(x_d - x), \qquad \dot y = \dot y_d + K_p\,(y_d - y), \qquad \dot\theta = \dot\theta_d + K_p\,\text{norm}(\theta_d - \theta)$$

donde $\text{norm}(\cdot)$ lleva el ángulo a $[-\pi, \pi]$. Pasar de 179° a −179° es un error de 2°, no de 358°. Puedes usar `math.atan2(math.sin(e), math.cos(e))`.

Además, en el nivel 3 el nodo usa la orientación **real** $\theta$ (no la deseada) para pasar al marco del robot.

✏️ **Escribe la función.**

In [ ]:
KP = 1.0

def controlador(deseado, real):
    xd, yd, thd, dxd, dyd, dthd = deseado
    x, y, th = real
    # >>> ESCRIBE AQUI <<<
    raise NotImplementedError

In [ ]:
# Simulación sencilla SIN ROS: un robot "imperfecto" que solo logra el 93% de la
# velocidad lineal y el 95% de la de giro (parecido a lo que pasa en Gazebo).
def simular(lazo_cerrado, kp=1.0, dt=0.05):
    global KP
    KP = kp
    x = y = th = 0.0
    camino, errores = [], []
    for t in np.arange(0, DURACION, dt):
        d = tray(t)
        if lazo_cerrado:
            dx, dy, dth = controlador(d, (x, y, th))
            vx, vy, w = mundo_a_robot(th, dx, dy, dth)        # theta REAL
        else:
            vx, vy, w = mundo_a_robot(d[2], *d[3:])           # theta DESEADA
        vx, vy, w = 0.93 * vx, 0.93 * vy, 0.95 * w            # el robot no es perfecto
        x += (math.cos(th) * vx - math.sin(th) * vy) * dt
        y += (math.sin(th) * vx + math.cos(th) * vy) * dt
        th += w * dt
        camino.append((x, y)); errores.append(math.hypot(d[0] - x, d[1] - y))
    return np.array(camino), np.array(errores)


try:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
    a1.plot(X[:, 0], X[:, 1], 'k--', label='deseada')
    for nombre, cerrado, kp in [('lazo abierto', False, 0), ('lazo cerrado Kp=0.5', True, 0.5),
                                ('lazo cerrado Kp=2', True, 2.0)]:
        camino, err = simular(cerrado, kp)
        a1.plot(camino[:, 0], camino[:, 1], label=nombre)
        a2.plot(np.arange(len(err)) * 0.05, err, label=f'{nombre} (RMS {np.sqrt(np.mean(err**2)):.3f} m)')
    a1.set_aspect('equal'); a1.legend(fontsize=8); a1.grid(alpha=0.3); a1.set_title('Trayectoria')
    a2.set_xlabel('t [s]'); a2.set_ylabel('error [m]'); a2.legend(fontsize=8); a2.grid(alpha=0.3)
    plt.show()
except NotImplementedError:
    print('⏳ Necesitas mundo_a_robot() y controlador()')

### En el simulador

1. Copia `controlador` (y `KP`) a `modelo.py`.
2. Con el simulador en `nivel:=2` o `nivel:=3` (son el mismo modo): `ros2 run taller_omni reiniciar`, luego `graficar` y `ros2 run taller_omni nivel3`.
3. Compara el gráfico con el del nivel 2: el error RMS debería bajar de varios centímetros a ~1 cm.

**Preguntas**
- Prueba `KP = 0`, `0.5`, `3`, `10`. ¿Qué pasa con un $K_p$ muy grande? *(pista: el lazo corre a 20 Hz)*
- En un robot real no existe `/taller/pose_real`. ¿De dónde sacarías la pose? *(odometría de ruedas, IMU, lidar + mapa…)*

## 7. Retos y lo que viene

**Retos** (en `modelo.py`; la trayectoria debe caber en la jaula):
1. Resuelve tu misma forma con **otro modo** de orientación. ¿Cuál sería imposible para un robot diferencial?
2. Si tu forma es un polígono, el robot "frena de golpe" en cada esquina. Suaviza las esquinas con un arco y compara el error.
3. **Ruta por puntos**: dada una lista de puntos cualquiera, une cada par con un segmento recto a velocidad constante. Eso es **interpolación lineal**, la base de la planificación.

**¿Y la planificación?** En este taller tú decides la trayectoria. En un sistema completo:
- un **planificador** (por ejemplo **A\***) busca los puntos de paso sobre el **mapa** construido con el lidar, evitando obstáculos;
- un **interpolador** convierte esos puntos en una curva suave con velocidades;
- un **controlador** como el de tu nivel 3 la sigue.

En ROS 2 eso lo hace **Nav2**, y es el siguiente paso.

## Problemas frecuentes

| Síntoma | Causa probable |
|---|---|
| `Package 'taller_omni' not found` | falta `source install/setup.bash` en esa terminal |
| El nodo se queda en *"Esperando la pose del robot"* | el simulador no está corriendo |
| `Nivel X: completa ... en modelo.py` | esa función todavía tiene el `raise NotImplementedError` |
| Nivel 2/3: el robot no se mueve | lanzaste el simulador con `nivel:=1`; relánzalo con `nivel:=2` |
| El robot sale de la jaula | no lo reiniciaste antes del intento: `ros2 run taller_omni reiniciar` |
| Cambié `modelo.py` y no pasa nada | detén el nodo (Ctrl+C) y vuelve a ejecutarlo |